Tratamento dos Dados (Camada Trusted)

Nesta etapa, os dados da camada RAW são lidos e preparados para uso analítico.

O tratamento realizado foi:

**Bairros**
- Validação dos tipos das colunas
- Verificação de possíveis áreas iguais a zero

**Concorrentes**
- Conversão de `codigo_bairro` para tipo inteiro anulável (Int64)
- Garantia de tipagem correta das colunas

**Eventos**
- Remoção de registros duplicados (chave composta: cliente + concorrente + datetime)
- Conversão da coluna `datetime` para tipo datetime
- Padronização dos tipos das colunas

**População**
- Conversão da coluna `populacao` para inteiro

Após o tratamento, os dados são armazenados na camada `trusted_silver` em formato Parquet.

LER DO RAW_BRONZE

In [ ]:
# ============================================================
# Configuração inicial e ingestão de dados - Camada Raw/Bronze
# Objetivo:
#   Definir o caminho base dos arquivos brutos e realizar a
#   leitura dos datasets para DataFrames do pandas.
# ============================================================

# 1) Importação de bibliotecas
# ------------------------------------------------------------
# pathlib.Path: manipulação segura e portátil de caminhos
# pandas: leitura e manipulação de dados tabulares

from pathlib import Path
import pandas as pd


# 2) Definição do caminho base da camada Raw/Bronze
# ------------------------------------------------------------
# Centraliza o diretório onde os arquivos brutos estão armazenados.
# Boa prática:
# - Evitar caminhos fixos absolutos
# - Facilitar portabilidade do projeto

RAW_PATH = Path("../data/raw_bronze")


# 3) Leitura dos arquivos
# ------------------------------------------------------------
# Cada dataset é carregado em um DataFrame específico.

# Base de bairros
df_bairros = pd.read_csv(RAW_PATH / "bairros.csv")

# Base de concorrentes
df_conc = pd.read_csv(RAW_PATH / "concorrentes.csv")

# Base de eventos de fluxo
df_eventos = pd.read_csv(RAW_PATH / "eventos_de_fluxo.csv")

# Base de população (formato JSON)
# Caso o JSON possua estrutura aninhada, pode ser necessário
# utilizar pd.json_normalize() posteriormente.
df_pop = pd.read_json(RAW_PATH / "populacao.json")


BAIRROS

In [2]:
# ============================================================
# Função: tratar_bairros
# Objetivo:
#   Validar integridade e padronizar tipagem da base de bairros,
#   garantindo consistência antes do uso analítico.
# ============================================================

def tratar_bairros(df_bairros):
    """
    Realiza validações e padronizações na base de bairros.

    Validações:
        - Verifica existência de área igual a zero
        - Verifica duplicidade da chave 'codigo'

    Padronizações:
        - Garante tipagem adequada das colunas

    Retorna:
        DataFrame tratado e padronizado.
    """

    print("="*60)
    print("VALIDAÇÃO E PADRONIZAÇÃO - BAIRROS")
    print("="*60)

    # ------------------------------------------------------------
    # Estado inicial do schema
    # -----------------------------------------------------------
    print("\nTipos antes:")
    print(df_bairros.dtypes)

    # =========================
    # VALIDAÇÕES
    # =========================

    # Verificar registros com área igual a zero
    # Boa prática: pode indicar erro de cadastro ou ausência de informação
    area_zero = (df_bairros["area"] == 0).sum()
    print(f"\nBairros com área igual a zero: {area_zero}")

    # Verificar duplicidade da chave primária
    # Boa prática: chave deve possuir 100% de unicidade
    chave_duplicada = df_bairros["codigo"].duplicated().sum()
    print(f"Códigos duplicados: {chave_duplicada}")

    # =========================
    # PADRONIZAÇÃO DE TIPOS
    # =========================
    # Boa prática:
    # - Forçar tipos explícitos evita problemas futuros em joins,
    #   agregações e exportações.

    df_bairros["codigo"] = df_bairros["codigo"].astype(int)
    df_bairros["area"] = df_bairros["area"].astype(float)
    df_bairros["municipio"] = df_bairros["municipio"].astype(str)
    df_bairros["uf"] = df_bairros["uf"].astype(str)

    # ------------------------------------------------------------
    # Estado final do schema
    # ------------------------------------------------------------

    print("\nTipos depois:")
    print(df_bairros.dtypes)

    # ------------------------------------------------------------
    # Resumo final de validação
    # ------------------------------------------------------------

    print("\nResumo final:")
    print(f"Total de registros: {df_bairros.shape[0]}")
    print(f"Área zero encontrada? {'Sim' if area_zero > 0 else 'Não'}")
    print(f"Chave duplicada encontrada? {'Sim' if chave_duplicada > 0 else 'Não'}")

    return df_bairros
# Aplicação da função
df_bairros = tratar_bairros(df_bairros)

VALIDAÇÃO E PADRONIZAÇÃO - BAIRROS

Tipos antes:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Bairros com área igual a zero: 0
Códigos duplicados: 0

Tipos depois:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Resumo final:
Total de registros: 133
Área zero encontrada? Não
Chave duplicada encontrada? Não


CONCORRENTES

In [3]:
# ============================================================
# Função: tratar_concorrentes_tipagem
# Objetivo:
#   Validar e padronizar tipagem da base de concorrentes,
#   garantindo consistência para análises e joins futuros.
# ============================================================

def tratar_concorrentes_tipagem(df_conc: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza validações e ajustes de tipagem na base de concorrentes.

    Validações realizadas:
        - Conversão da coluna 'codigo_bairro' para inteiro nullable
        - Verificação de valores nulos em 'codigo_bairro'
        - Validação da tipagem da coluna 'faixa_preco'
        - Inspeção de valores únicos em 'faixa_preco'

    Retorna:
        DataFrame validado e padronizado.
    """

    print("=" * 60)
    print("VALIDAÇÃO E PADRONIZAÇÃO - CONCORRENTES")
    print("=" * 60)

    # ============================================================
    # 1) Validação e padronização de codigo_bairro
    # ============================================================
    print("\n[ codigo_bairro ]")
    print("Tipo antes:", df_conc["codigo_bairro"].dtype)

    # Boa prática:
    # - Utilizar tipo "Int64" (nullable) ao invés de int64
    # - Permite manter valores nulos sem gerar erro

    try:
        df_conc["codigo_bairro"] = df_conc["codigo_bairro"].astype("Int64")
        print("Tipo depois:", df_conc["codigo_bairro"].dtype)
    except Exception as e:
        print("Erro ao converter codigo_bairro:", e)

    # Verificação de valores nulos
    null_bairro = df_conc["codigo_bairro"].isnull().sum()
    print("Quantidade de null em codigo_bairro:", null_bairro)


    # ============================================================
    # 2) Validação de faixa_preco
    # ============================================================
    print("\n[ faixa_preco ]")
    print("Tipo atual:", df_conc["faixa_preco"].dtype)

    # Inspeção de valores únicos
    # Útil para identificar inconsistências (ex: texto misturado com números)
    valores_unicos = df_conc["faixa_preco"].unique()
    print("Valores únicos:", valores_unicos)

    # Verificação se a coluna é numérica
    is_numeric = pd.api.types.is_numeric_dtype(df_conc["faixa_preco"])
    print("É numérico válido?", is_numeric)

    if not is_numeric:
        print("Faixa_preco não é numérico. Avaliar necessidade de conversão ou categorização.")


    # ============================================================
    # 3) Resumo final
    # ============================================================
    print("\nTotal de registros:", f"{df_conc.shape[0]:,}")

    return df_conc


# Aplicação da função
df_conc = tratar_concorrentes_tipagem(df_conc)

VALIDAÇÃO E PADRONIZAÇÃO - CONCORRENTES

[ codigo_bairro ]
Tipo antes: float64
Tipo depois: Int64
Quantidade de null em codigo_bairro: 2752

[ faixa_preco ]
Tipo atual: int64
Valores únicos: [2 0 3 1 4]
É numérico válido? True

Total de registros: 4,202


EVENTOS

In [ ]:
# ============================================================
# Função: remover_duplicados_validado
# Objetivo:
#   Validar e remover registros duplicados com base em uma
#   chave composta (subset), mantendo transparência do impacto.
# ============================================================

# Boa prática:
# - Medir impacto antes e depois da remoção
# - Permitir visualização opcional dos registros afetados

def remover_duplicados_validado(df, subset, mostrar_duplicados=True):
    print(" VALIDAÇÃO DE DUPLICADOS ".center(70, "-"))
    
    # ------------------------------------------------------------
    # 1) Estado inicial da base
    # ------------------------------------------------------------
    # Captura o total de registros antes da limpeza
    total_antes = df.shape[0]
    
    # ------------------------------------------------------------
    # 2) Identificação de duplicados
    # ------------------------------------------------------------
    # duplicated(keep=False):
    # - Marca todas as ocorrências duplicadas
    # - Permite visualizar o grupo completo de registros repetidos
    duplicados = df[df.duplicated(subset=subset, keep=False)]
    qtd_duplicados = duplicados.shape[0]
    
    print(f"Total antes: {total_antes:,}")
    print(f"Registros duplicados encontrados: {qtd_duplicados:,}")
    
    # ------------------------------------------------------------
    # 3) Visualização opcional para auditoria
    # ------------------------------------------------------------
    # Boa prática:
    # - Exibir duplicados antes de remover
    # - Facilita validação manual e rastreabilidade
    if mostrar_duplicados and qtd_duplicados > 0:
        print("\nDuplicados identificados:")
        display(duplicados.sort_values(subset))
    
    # ------------------------------------------------------------
    # 4) Remoção dos duplicados
    # ------------------------------------------------------------
    # drop_duplicates mantém a primeira ocorrência por padrão.
    # Caso necessário, pode-se definir critério adicional
    # (ex: ordenar por data antes de remover).
    df_limpo = df.drop_duplicates(subset=subset)
    
    # ------------------------------------------------------------
    # 5) Medição de impacto
    # ------------------------------------------------------------
    total_depois = df_limpo.shape[0]
    removidos = total_antes - total_depois
    
    print(f"\nTotal depois: {total_depois:,}")
    print(f"Registros removidos: {removidos:,}")
    
    print("-" * 70)
    
    return df_limpo


# Aplicação da função
# - Garantir que essas colunas representem a regra de negócio

df_eventos = remover_duplicados_validado(
    df_eventos,
    subset=["codigo", "codigo_concorrente", "datetime"]
)

---------------------- VALIDAÇÃO DE DUPLICADOS -----------------------
Total antes: 209,801
Registros duplicados encontrados: 0

Total depois: 209,801
Registros removidos: 0
----------------------------------------------------------------------


In [9]:
# ============================================================
# Conversão e validação da coluna datetime
# Objetivo:
#   Garantir tipagem correta para datetime, identificar erros
#   de conversão e validar intervalo temporal dos dados.
# ============================================================

print("=" * 70)
print("CONVERSÃO E VALIDAÇÃO - DATETIME")
print("=" * 70)

# ------------------------------------------------------------
# 1) Verificação do tipo antes da conversão
# ------------------------------------------------------------
# Boa prática:
# - Sempre validar o dtype antes de converter
# - Evita conversões desnecessárias ou mascaramento de erros

print("\nTipo antes:", df_eventos["datetime"].dtype)


# ------------------------------------------------------------
# 2) Conversão para datetime
# ------------------------------------------------------------
# pd.to_datetime:
# - format="ISO8601": otimiza conversão para padrão ISO
# - errors="coerce": converte valores inválidos em NaT
#
# Boa prática:
# - Usar errors="coerce" para não interromper o pipeline
# - Posteriormente medir impacto dos NaT gerados

df_eventos["datetime"] = pd.to_datetime(
    df_eventos["datetime"],
    format="ISO8601",
    errors="coerce"
)


# ------------------------------------------------------------
# 3) Verificação do tipo após conversão
# ------------------------------------------------------------
print("Tipo depois:", df_eventos["datetime"].dtype)


# ------------------------------------------------------------
# 4) Validação de valores inválidos
# ------------------------------------------------------------
# NaT (Not a Time) indica falha na conversão
# Boa prática:
# - Sempre medir quantidade de NaT após conversão

nat = df_eventos["datetime"].isna().sum()
print("Valores NaT após conversão:", nat)


# ------------------------------------------------------------
# 5) Validação do intervalo temporal
# ------------------------------------------------------------
# Se houver pelo menos uma data válida,
# calcula intervalo mínimo e máximo.

if nat < len(df_eventos):
    print("Data mínima:", df_eventos["datetime"].min())
    print("Data máxima:", df_eventos["datetime"].max())
else:
    print("Todas as datas ficaram inválidas!")


# ------------------------------------------------------------
# 6) Amostra para inspeção manual
# ------------------------------------------------------------
# Boa prática:
# - Visualizar amostra após transformação crítica

print("\nAmostra:")
display(df_eventos["datetime"].head())

CONVERSÃO E VALIDAÇÃO - DATETIME

Tipo antes: datetime64[ns]
Tipo depois: datetime64[ns]
Valores NaT após conversão: 0
Data mínima: 2017-05-31 22:34:36.923000
Data máxima: 2017-07-31 20:51:01.189000

Amostra:


0   2017-07-27 09:51:02.000
1   2017-06-24 14:00:26.405
2   2017-07-06 21:51:11.056
3   2017-07-02 14:27:09.316
4   2017-07-15 03:03:29.731
Name: datetime, dtype: datetime64[ns]

In [11]:
# ============================================================
# Validação de tipos esperados - df_eventos
# Objetivo:
#   Garantir que colunas críticas estejam com a tipagem correta
#   antes de operações analíticas ou integrações.
# ============================================================

print(" VALIDAÇÃO DE TIPOS ".center(70, "-"))

# ------------------------------------------------------------
# 1) Definição do schema esperado
# ------------------------------------------------------------
# Boa prática:
# - Centralizar definição de tipos esperados
# - Facilita auditoria e manutenção
# - Deixa explícita a regra de negócio do dataset

tipos_esperados = {
    "codigo": "object",
    "codigo_concorrente": "int64",
    "datetime": "datetime64[ns]"
}

# ------------------------------------------------------------
# 2) Comparação entre tipo atual e tipo esperado
# ------------------------------------------------------------
# Boa prática:
# - Converter dtype para string antes da comparação
# - Facilita padronização da validação
# - Evita erro por comparação direta de objetos dtype

for coluna, tipo_esperado in tipos_esperados.items():
    
    # Tipo atual da coluna
    tipo_atual = str(df_eventos[coluna].dtype)
    
    # Status da validação
    status = "OK" if tipo_atual == tipo_esperado else "ERRO"
    
    # Exibição formatada para melhor leitura
    print(f"{coluna:<20} -> {tipo_atual:<20} [{status}]")

------------------------- VALIDAÇÃO DE TIPOS -------------------------
codigo               -> object               [OK]
codigo_concorrente   -> int64                [OK]
datetime             -> datetime64[ns]       [OK]


POPULAÇÃO

In [17]:
# ============================================================
# Função: tratar_populacao
# Objetivo:
#   Validar estado inicial da base de população e converter
#   a coluna "populacao" para inteiro anulável (Int64),
#   preservando valores nulos.
# ============================================================

def tratar_populacao(df_pop):
    print("=" * 60)
    print("ESTADO INICIAL")
    print("=" * 60)
    
    # ------------------------------------------------------------
    # 1) Diagnóstico inicial
    # ------------------------------------------------------------
    # Boa prática:
    # - Sempre registrar estado antes de transformação
    # - Facilita auditoria e comparação
    
    print("\nShape inicial:", df_pop.shape)
    
    print("\nTipos antes:")
    print(df_pop.dtypes)
    
    print("\nNulls antes:")
    print(df_pop.isnull().sum())
    
    print("\nValores únicos de populacao antes:")
    print(df_pop["populacao"].unique()[:10])
    
    
    # ============================================================
    # 2) Tratamento
    # ============================================================
    # Boa prática:
    # - Trabalhar sobre cópia do DataFrame
    # - Evita modificar objeto original inadvertidamente
    
    df_tratado = df_pop.copy()
    
    # Conversão para inteiro anulável
    # Int64 (I maiúsculo) permite valores nulos
    # Diferente de int64 tradicional, que não aceita NaN
    df_tratado["populacao"] = df_tratado["populacao"].astype("Int64")
    
    
    # ------------------------------------------------------------
    # 3) Diagnóstico após tratamento
    # ------------------------------------------------------------
    # Boa prática:
    # - Validar se transformação alterou volume de registros
    # - Confirmar tipagem correta
    # - Garantir que nulls foram preservados
    
    print("\n" + "=" * 60)
    print("ESTADO APÓS TRATAMENTO")
    print("=" * 60)
    
    print("\nShape final:", df_tratado.shape)
    
    print("\nTipos depois:")
    print(df_tratado.dtypes)
    
    print("\nNulls depois:")
    print(df_tratado.isnull().sum())
    
    print("\nValores únicos de populacao depois:")
    print(df_tratado["populacao"].unique()[:10])
    
    return df_tratado


# Aplicação da função
df_pop = tratar_populacao(df_pop)

ESTADO INICIAL

Shape inicial: (133, 2)

Tipos antes:
codigo       int64
populacao    Int64
dtype: object

Nulls antes:
codigo       0
populacao    1
dtype: int64

Valores únicos de populacao antes:
<IntegerArray>
[8717, 5764, 1195, 17840, 1252, 206, 1809, 105720, 2334, 1064]
Length: 10, dtype: Int64

ESTADO APÓS TRATAMENTO

Shape final: (133, 2)

Tipos depois:
codigo       int64
populacao    Int64
dtype: object

Nulls depois:
codigo       0
populacao    1
dtype: int64

Valores únicos de populacao depois:
<IntegerArray>
[8717, 5764, 1195, 17840, 1252, 206, 1809, 105720, 2334, 1064]
Length: 10, dtype: Int64


SALVANDO EM PARQUET

In [ ]:
# ============================================================
# Persistência dos dados - Camada Trusted/Silver
# Objetivo:
#   Salvar os DataFrames tratados em formato Parquet
#   na camada trusted (silver) do projeto.
# ============================================================

print(" SALVANDO EM PARQUET ".center(70, "-"))

# ------------------------------------------------------------
# 1) Definição do caminho da camada Trusted/Silver
# ------------------------------------------------------------
# Boa prática:
# - Separar camadas do pipeline (raw → trusted → refined)
# - Centralizar caminhos para facilitar manutenção

TRUSTED_PATH = Path("../data/trusted_silver")


# ------------------------------------------------------------
# 2) Garantir existência do diretório
# ------------------------------------------------------------
# mkdir(parents=True, exist_ok=True):
# - Cria a pasta caso não exista
# - Evita erro caso diretório já esteja criado
# - parents=True permite criar estrutura completa

TRUSTED_PATH.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 3) Salvamento em formato Parquet
# ------------------------------------------------------------
# Boa prática:
# - Utilizar Parquet por ser colunar e eficiente
# - Reduz espaço em disco
# - Melhora performance de leitura futura
# - index=False evita salvar índice desnecessário

df_bairros.to_parquet(TRUSTED_PATH / "bairros.parquet", index=False)
df_conc.to_parquet(TRUSTED_PATH / "concorrentes.parquet", index=False)
df_eventos.to_parquet(TRUSTED_PATH / "eventos.parquet", index=False)
df_pop.to_parquet(TRUSTED_PATH / "populacao.parquet", index=False)


# ------------------------------------------------------------
# 4) Confirmação de persistência
# ------------------------------------------------------------
# Boa prática:
# - Informar local de saída
# - Facilita rastreabilidade

print("Arquivos salvos com sucesso em:", TRUSTED_PATH)

------------------------ SALVANDO EM PARQUET -------------------------
Arquivos salvos com sucesso em: ../data/trusted_silver


Código para testar leitura da TRUSTED

In [19]:
# ============================================================
# Teste de leitura - Camada Trusted/Silver
# Objetivo:
#   Validar se os arquivos salvos em Parquet foram persistidos
#   corretamente e mantiveram estrutura e tipagem esperadas.
# ============================================================

print(" TESTE DE LEITURA - TRUSTED_SILVER ".center(70, "-"))

# ------------------------------------------------------------
# 1) Definição do caminho da camada Trusted
# ------------------------------------------------------------
# Boa prática:
# - Reutilizar variável centralizada de caminho
# - Evitar hardcode repetido no projeto

TRUSTED_PATH = Path("../data/trusted_silver")


# ------------------------------------------------------------
# 2) Leitura dos arquivos Parquet
# ------------------------------------------------------------
# Boa prática:
# - Validar leitura imediatamente após escrita
# - Confirmar que tipos foram preservados

bairros = pd.read_parquet(TRUSTED_PATH / "bairros.parquet")
concorrentes = pd.read_parquet(TRUSTED_PATH / "concorrentes.parquet")
eventos = pd.read_parquet(TRUSTED_PATH / "eventos.parquet")
populacao = pd.read_parquet(TRUSTED_PATH / "populacao.parquet")


# ------------------------------------------------------------
# 3) Organização dos datasets para validação iterativa
# ------------------------------------------------------------
# Boa prática:
# - Utilizar dicionário para evitar repetição de código
# - Facilitar validação padronizada

datasets = {
    "BAIRROS": bairros,
    "CONCORRENTES": concorrentes,
    "EVENTOS": eventos,
    "POPULACAO": populacao
}


# ------------------------------------------------------------
# 4) Validação estrutural de cada dataset
# ------------------------------------------------------------
# Para cada DataFrame:
# - Exibe shape
# - Exibe tipagem
# - Mostra amostra inicial
#
# Boa prática:
# - Confirmar integridade após persistência
# - Verificar se tipos como datetime e Int64 foram preservados

for nome, df in datasets.items():
    print("\n" + f" {nome} ".center(70, "-"))
    
    print("Shape:", df.shape)
    
    print("Tipos:")
    print(df.dtypes)
    
    print("\nTop 5:")
    display(df.head())

----------------- TESTE DE LEITURA - TRUSTED_SILVER ------------------

------------------------------ BAIRROS -------------------------------
Shape: (133, 5)
Tipos:
codigo         int64
nome          object
municipio     object
uf            object
area         float64
dtype: object

Top 5:


,codigo,nome,municipio,uf,area
0,355620110,Observatório,Valinhos,SP,68.000900
1,3519071024,Rp 6-24,Hortolândia,SP,0.981768
2,3536505002,Jardim De Itapoan,Paulínia,SP,0.808537
3,3519071026,Rp 6-26,Hortolândia,SP,2.211080
4,3536505001,Nova Paulínia,Paulínia,SP,0.386199



---------------------------- CONCORRENTES ----------------------------
Shape: (4202, 8)
Tipos:
codigo            int64
nome             object
categoria        object
faixa_preco       int64
endereco         object
municipio        object
uf               object
codigo_bairro     Int64
dtype: object

Top 5:


,codigo,nome,categoria,faixa_preco,endereco,municipio,uf,codigo_bairro
0,431962533652067,Boizão Lanches,Bar,2,13190-000 Monte Mor,Monte Mor,SP,<NA>
1,1663855903830869,Bar do Serjão,Bar,0,"Rua das Dracenas, Americana",Americana,SP,<NA>
2,567824576564110,Recanto Do Kuca,Restaurant,0,Jarinu,Jarinu,SP,<NA>
3,202740866540615,Dedé Abelhuda,Grocery Store,0,"SP, 13150000 Cosmópolis",Cosmópolis,SP,<NA>
4,1784900838394305,Tenshi Sushi Boteco Itu,Sushi Restaurant,3,"Av. Plaza, 170, 13302-100 Itu",Itu,SP,<NA>



------------------------------ EVENTOS -------------------------------
Shape: (209801, 3)
Tipos:
codigo                        object
datetime              datetime64[ns]
codigo_concorrente             int64
dtype: object

Top 5:


,codigo,datetime,codigo_concorrente
0,oMn07h1bJYV0Wdx+RTzsDcT8JQlT7QXc7q8A/4y+cO5gBQ...,2017-07-27 09:51:02.000,650509405109544
1,iZuQeTd9am+qfaiqnn5kkixogIbwN0nY2gtMwZqH9bFqph...,2017-06-24 14:00:26.405,650509405109544
2,iIMvwWSnQoW0aqQlcvjc8A6LvjkRX1HLppdkdQZapPVVv7...,2017-07-06 21:51:11.056,650509405109544
3,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-02 14:27:09.316,650509405109544
4,jYaJZuli3fVSNDOs60if9VObBmCOoJP2x9H3IZP+6FsMRk...,2017-07-15 03:03:29.731,650509405109544



----------------------------- POPULACAO ------------------------------
Shape: (133, 2)
Tipos:
codigo       int64
populacao    Int64
dtype: object

Top 5:


,codigo,populacao
0,355620110,8717
1,3519071024,5764
2,3536505002,1195
3,3519071026,17840
4,3536505001,1252
